# Image Synthesis from Semantic Labeling

A complete from-scratch implementation of a **Conditional GAN (Pix2Pix-style)** for generating photorealistic images from semantic label maps.

**Architecture:**
```
Semantic Label Map ──► U-Net Generator ──► Synthesized Image
                            ▲
                            │ Adversarial + L1 Loss
                            ▼
                      PatchGAN Discriminator ◄── Real Image
```

**Key components (all written from scratch):**
- **U-Net Generator** — Encoder-decoder with skip connections
- **PatchGAN Discriminator** — Classifies 70×70 patches as real/fake
- **Losses** — Adversarial (BCE), L1 reconstruction, feature matching

---
## 1. Setup & Imports

In [ ]:
import os
import sys
import csv
import time
import tarfile
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:    {DEVICE}")

---
## 2. Configuration

All hyperparameters are defined here. Adjust as needed.

In [2]:
class Config:
    """All hyperparameters and paths in one place."""

    # ── Paths ──────────────────────────────────────────────────────
    dataset_dir   = "./data/facades"
    output_dir    = "./outputs"

    # ── Data ───────────────────────────────────────────────────────
    image_size    = 128
    input_nc      = 3       # channels in semantic label input
    output_nc     = 3       # channels in generated output
    batch_size    = 8
    num_workers   = 2
    persistent_workers = True
    direction     = "BtoA"  # AtoB = label→photo, BtoA = photo→label

    # ── Generator (U-Net) ──────────────────────────────────────────
    ngf           = 64      # base filter count
    unet_depth    = 7       # downsampling blocks (8 for 256×256)
    gen_dropout   = 0.5     # dropout in first 3 decoder blocks

    # ── Discriminator (PatchGAN) ───────────────────────────────────
    ndf           = 64      # base filter count
    n_layers_d    = 3       # intermediate conv layers

    # ── Training ───────────────────────────────────────────────────
    epochs        = 200
    lr_g          = 2e-4
    lr_d          = 2e-4
    beta1         = 0.5
    beta2         = 0.999
    lambda_l1     = 100.0   # L1 reconstruction weight
    lambda_fm     = 10.0    # feature matching weight
    use_feature_matching = False
    n_critic      = 1       # D updates per G update

    # ── Logging ────────────────────────────────────────────────────
    save_interval   = 10    # checkpoint every N epochs
    sample_interval = 5     # save visuals every N epochs

    # ── Device ─────────────────────────────────────────────────────
    device = DEVICE

cfg = Config()

# Create output directories
cfg.checkpoint_dir = Path(cfg.output_dir) / "checkpoints"
cfg.sample_dir     = Path(cfg.output_dir) / "samples"
cfg.log_dir        = Path(cfg.output_dir) / "logs"
cfg.inference_dir  = Path(cfg.output_dir) / "inference"

for d in [cfg.checkpoint_dir, cfg.sample_dir, cfg.log_dir, cfg.inference_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Config ready ✓")

Config ready ✓


---
## 3. U-Net Generator (from scratch)

The generator follows an encoder-decoder architecture with **skip connections** at every resolution level:

- **Encoder**: 8 downsampling blocks (Conv → BatchNorm → LeakyReLU)
- **Decoder**: 7 upsampling blocks (ConvTranspose → BatchNorm → Dropout → ReLU) + skip concatenation
- **Output**: ConvTranspose → Tanh (maps to [-1, 1])

In [19]:
class EncoderBlock(nn.Module):
    """Single downsampling block: Conv → [BN] → LeakyReLU."""

    def __init__(self, in_channels, out_channels, use_batchnorm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=4, stride=2, padding=1,
                      bias=not use_batchnorm)
        ]
        if use_batchnorm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    """Single upsampling block: ConvTranspose → BN → [Dropout] → ReLU."""

    def __init__(self, in_channels, out_channels, use_dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels,
                               kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        if use_dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        """Forward with skip connection concatenation."""
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """
    U-Net Generator for conditional image synthesis.

    Converts a semantic label map into a photorealistic image using a
    symmetric encoder-decoder with skip connections at every spatial level.

    Args:
        input_nc:  Number of input channels (label map).
        output_nc: Number of output channels (synthesized image).
        ngf:       Base number of filters (doubled at each encoder level).
        depth:     Number of downsampling steps (8 for 256×256 input).
        dropout:   Dropout probability in the first 3 decoder blocks.
    """

    def __init__(self, input_nc=3, output_nc=3, ngf=64, depth=8, dropout=0.5):
        super().__init__()
        assert depth >= 4, "U-Net depth must be >= 4"
        self.depth = depth

        # ── Build Encoder ──────────────────────────────────────────
        self.encoders = nn.ModuleList()
        encoder_channels = []

        in_ch = input_nc
        for i in range(depth):
            if i == 0:
                out_ch = ngf
                use_bn = False   # first block: no batchnorm
            elif i == depth - 1:
                out_ch = ngf * 8
                use_bn = False
            elif i <= 3:
                out_ch = ngf * min(2 ** i, 8)
                use_bn = True
            else:
                out_ch = ngf * 8  # cap at ngf*8
                use_bn = True

            self.encoders.append(EncoderBlock(in_ch, out_ch, use_batchnorm=use_bn))
            encoder_channels.append(out_ch)
            in_ch = out_ch

        # ── Build Decoder ──────────────────────────────────────────
        self.decoders = nn.ModuleList()

        in_ch = encoder_channels[-1]  # bottleneck
        for i in range(depth - 1):
            enc_idx = depth - 2 - i
            skip_ch = encoder_channels[enc_idx]
            use_drop = (i < 3) and (dropout > 0)

            if i == 0:
                out_ch = skip_ch
                self.decoders.append(DecoderBlock(in_ch, out_ch, use_dropout=use_drop))
            else:
                out_ch = skip_ch
                self.decoders.append(DecoderBlock(in_ch, out_ch, use_dropout=use_drop))
            in_ch = out_ch + skip_ch  # after concatenation

        # ── Final output layer ─────────────────────────────────────
        self.final = nn.Sequential(
            nn.ConvTranspose2d(in_ch, output_nc, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        classname = m.__class__.__name__
        if classname.find("Conv") != -1:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif classname.find("BatchNorm") != -1:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

    def forward(self, x):
        # Encoder — save features for skip connections
        encoder_features = []
        h = x
        for encoder in self.encoders:
            h = encoder(h)
            encoder_features.append(h)

        # Decoder — with skip connections
        h = encoder_features[-1]
        for i, decoder in enumerate(self.decoders):
            skip = encoder_features[self.depth - 2 - i]
            h = decoder(h, skip)

        return self.final(h)


# ── Test ───────────────────────────────────────────────────────────
G = UNetGenerator(3, 3, 64, 8).to("cpu")
x = torch.randn(1, 3, 256, 256)
out = G(x)
print(f"Generator: {x.shape} → {out.shape}")
print(f"Parameters: {sum(p.numel() for p in G.parameters()):,}")
del G, x, out
print("✓ Generator test passed")

Generator: torch.Size([1, 3, 256, 256]) → torch.Size([1, 3, 256, 256])
Parameters: 54,414,531
✓ Generator test passed


---
## 4. PatchGAN Discriminator (from scratch)

The discriminator classifies overlapping **70×70 patches** as real or fake. It is conditioned on both the semantic label map and the image (real or generated), concatenated along the channel dimension.

Output is a spatial map `[B, 1, H', W']` — each value judges one receptive field.

In [20]:
class DiscriminatorBlock(nn.Module):
    """Conv → [BN] → LeakyReLU block."""

    def __init__(self, in_channels, out_channels, stride=2, use_batchnorm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=4, stride=stride, padding=1,
                      bias=not use_batchnorm)
        ]
        if use_batchnorm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN Discriminator for conditional GAN.

    Takes [label_map | image] concatenated input and outputs a spatial
    map of real/fake predictions. Also returns intermediate feature maps
    for the optional feature matching loss.

    Args:
        input_nc:  Channels in the label map.
        output_nc: Channels in the image.
        ndf:       Base number of filters.
        n_layers:  Number of intermediate conv layers.
    """

    def __init__(self, input_nc=3, output_nc=3, ndf=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        total_input_nc = input_nc + output_nc

        self.blocks = nn.ModuleList()

        # First layer: no batchnorm
        self.blocks.append(
            DiscriminatorBlock(total_input_nc, ndf, stride=2, use_batchnorm=False)
        )

        # Intermediate layers
        in_ch = ndf
        for i in range(1, n_layers):
            out_ch = ndf * min(2 ** i, 8)
            self.blocks.append(DiscriminatorBlock(in_ch, out_ch, stride=2, use_batchnorm=True))
            in_ch = out_ch

        # Penultimate layer: stride=1
        out_ch = ndf * min(2 ** n_layers, 8)
        self.blocks.append(DiscriminatorBlock(in_ch, out_ch, stride=1, use_batchnorm=True))
        in_ch = out_ch

        # Final 1-channel prediction (no BN, no activation)
        self.final = nn.Conv2d(in_ch, 1, kernel_size=4, stride=1, padding=1)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        classname = m.__class__.__name__
        if classname.find("Conv") != -1:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif classname.find("BatchNorm") != -1:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0)

    def forward(self, label_map, image):
        """
        Args:
            label_map: [B, input_nc, H, W]
            image:     [B, output_nc, H, W] (real or fake)

        Returns:
            features:   List of intermediate feature maps (for feature matching).
            prediction: [B, 1, H', W'] real/fake map.
        """
        x = torch.cat([label_map, image], dim=1)

        features = []
        for block in self.blocks:
            x = block(x)
            features.append(x)

        prediction = self.final(x)
        return features, prediction


# ── Test ───────────────────────────────────────────────────────────
D = PatchGANDiscriminator(3, 3, 64, 3).to("cpu")
lbl = torch.randn(1, 3, 256, 256)
img = torch.randn(1, 3, 256, 256)
feats, pred = D(lbl, img)
print(f"Discriminator: label {lbl.shape} + image {img.shape} → pred {pred.shape}")
print(f"Feature maps: {[f.shape for f in feats]}")
print(f"Parameters: {sum(p.numel() for p in D.parameters()):,}")
del D, lbl, img, feats, pred
print("✓ Discriminator test passed")

Discriminator: label torch.Size([1, 3, 256, 256]) + image torch.Size([1, 3, 256, 256]) → pred torch.Size([1, 1, 30, 30])
Feature maps: [torch.Size([1, 64, 128, 128]), torch.Size([1, 128, 64, 64]), torch.Size([1, 256, 32, 32]), torch.Size([1, 512, 31, 31])]
Parameters: 2,768,705
✓ Discriminator test passed


---
## 5. Loss Functions (from scratch)

Three loss components, all implemented with raw PyTorch operations:

| Loss | Purpose | Weight |
|------|---------|--------|
| **GANLoss** | Adversarial — trains G to fool D | 1.0 |
| **L1ReconstructionLoss** | Pixel-level similarity to ground truth | λ_L1 = 100 |
| **FeatureMatchingLoss** | Match D's internal feature statistics | λ_FM = 10 |

In [21]:
class GANLoss(nn.Module):
    """
    Adversarial loss (vanilla BCE or LSGAN).

    Automatically creates target tensors of the correct shape
    filled with real_label or fake_label values.
    """

    def __init__(self, mode="vanilla", real_label=1.0, fake_label=0.0):
        super().__init__()
        assert mode in ("vanilla", "lsgan")
        self.mode = mode
        self.register_buffer("real_val", torch.tensor(real_label))
        self.register_buffer("fake_val", torch.tensor(fake_label))

    def _make_target(self, prediction, is_real):
        value = self.real_val if is_real else self.fake_val
        return value.expand_as(prediction)

    def forward(self, prediction, is_real):
        target = self._make_target(prediction, is_real)
        if self.mode == "vanilla":
            return F.binary_cross_entropy_with_logits(prediction, target)
        else:
            return F.mse_loss(prediction, target)


class L1ReconstructionLoss(nn.Module):
    """Weighted pixel-wise L1 distance between generated and target images."""

    def __init__(self, weight=100.0):
        super().__init__()
        self.weight = weight

    def forward(self, generated, target):
        return self.weight * torch.mean(torch.abs(generated - target))


class FeatureMatchingLoss(nn.Module):
    """
    L1 distance between discriminator intermediate features
    extracted from real vs. generated images.
    """

    def __init__(self, weight=10.0):
        super().__init__()
        self.weight = weight

    def forward(self, features_real, features_fake):
        loss = 0.0
        for fr, ff in zip(features_real, features_fake):
            loss += torch.mean(torch.abs(fr.detach() - ff))
        return self.weight * loss / len(features_real)


# ── Test ───────────────────────────────────────────────────────────
gan_loss = GANLoss("vanilla")
pred = torch.randn(2, 1, 30, 30)
print(f"GAN loss (real): {gan_loss(pred, True).item():.4f}")
print(f"GAN loss (fake): {gan_loss(pred, False).item():.4f}")

l1 = L1ReconstructionLoss(100.0)
print(f"L1 loss: {l1(torch.randn(2,3,64,64), torch.randn(2,3,64,64)).item():.4f}")

fm = FeatureMatchingLoss(10.0)
fr = [torch.randn(2, 64, 32, 32)]
ff = [torch.randn(2, 64, 32, 32)]
print(f"FM loss: {fm(fr, ff).item():.4f}")
print("✓ All loss tests passed")

GAN loss (real): 0.8113
GAN loss (fake): 0.7946
L1 loss: 112.5110
FM loss: 11.2634
✓ All loss tests passed


---
## 6. Dataset

Loads **paired images** where each file is a horizontally concatenated pair: `[image_A | image_B]`.

The Facades dataset can be auto-downloaded. For Cityscapes or custom data, place paired images in `data/<name>/train/`.

In [22]:
class PairedImageDataset(Dataset):
    """
    Dataset of paired images (label map ↔ photo).

    Each file is a side-by-side concatenation: [left | right].
    With direction='AtoB': left = input (label), right = target (photo).
    """

    def __init__(self, root_dir, image_size=256, direction="AtoB",
                 split="train", augment=True):
        super().__init__()
        self.image_size = image_size
        self.direction = direction
        self.augment = augment and (split == "train")

        self.root = Path(root_dir)
        search_dir = self.root / split if (self.root / split).exists() else self.root

        self.image_paths = sorted([
            p for p in search_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
        ])

        if len(self.image_paths) == 0:
            raise FileNotFoundError(f"No images found in {search_dir}")

        print(f"[Dataset] {len(self.image_paths)} paired images in '{search_dir}'")

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size), Image.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def _random_augment(self, img_a, img_b):
        if torch.rand(1).item() > 0.5:
            img_a = img_a.flip(-1)
            img_b = img_b.flip(-1)
        return img_a, img_b

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        w, h = img.size
        mid = w // 2

        img_a = self.transform(img.crop((0, 0, mid, h)))
        img_b = self.transform(img.crop((mid, 0, w, h)))

        if self.augment:
            img_a, img_b = self._random_augment(img_a, img_b)

        if self.direction == "AtoB":
            return img_a, img_b   # label, photo
        else:
            return img_b, img_a


def create_dataloaders(cfg):
    """Create train and (optional) val DataLoaders."""
    train_ds = PairedImageDataset(
        cfg.dataset_dir, cfg.image_size, cfg.direction, "train", augment=True
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True, persistent_workers=True
    )

    val_loader = None
    val_dir = Path(cfg.dataset_dir) / "val"
    if val_dir.exists() and any(val_dir.iterdir()):
        val_ds = PairedImageDataset(
            cfg.dataset_dir, cfg.image_size, cfg.direction, "val", augment=False
        )
        val_loader = DataLoader(
            val_ds, batch_size=cfg.batch_size, shuffle=False,
            num_workers=cfg.num_workers, pin_memory=True
        )

    return train_loader, val_loader


print("Dataset classes defined ✓")

Dataset classes defined ✓


---
## 7. Utility Functions

Visualization, checkpointing, and loss logging.

In [23]:
def denormalize(tensor):
    """Convert from [-1, 1] to [0, 1] for display."""
    return (tensor + 1.0) / 2.0


def show_sample_grid(label_maps, generated, real_images, max_samples=4, save_path=None):
    """Display / save a grid of [label | generated | real] triplets."""
    n = min(max_samples, label_maps.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = [axes]

    titles = ["Semantic Label", "Generated", "Ground Truth"]

    for i in range(n):
        images = [
            denormalize(label_maps[i]).cpu().permute(1, 2, 0).numpy().clip(0, 1),
            denormalize(generated[i]).cpu().permute(1, 2, 0).numpy().clip(0, 1),
            denormalize(real_images[i]).cpu().permute(1, 2, 0).numpy().clip(0, 1),
        ]
        for j in range(3):
            ax = axes[i][j] if n > 1 else axes[j]
            ax.imshow(images[j])
            if i == 0:
                ax.set_title(titles[j], fontsize=14)
            ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_checkpoint(generator, discriminator, opt_g, opt_d, epoch, path):
    torch.save({
        "epoch": epoch,
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "optimizer_g_state_dict": opt_g.state_dict(),
        "optimizer_d_state_dict": opt_d.state_dict(),
    }, path)


def load_checkpoint(path, generator, discriminator=None, opt_g=None, opt_d=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    generator.load_state_dict(ckpt["generator_state_dict"])
    if discriminator and "discriminator_state_dict" in ckpt:
        discriminator.load_state_dict(ckpt["discriminator_state_dict"])
    if opt_g and "optimizer_g_state_dict" in ckpt:
        opt_g.load_state_dict(ckpt["optimizer_g_state_dict"])
    if opt_d and "optimizer_d_state_dict" in ckpt:
        opt_d.load_state_dict(ckpt["optimizer_d_state_dict"])
    return ckpt.get("epoch", 0)


class LossLogger:
    """CSV logger for training losses."""

    def __init__(self, log_dir):
        self.log_path = Path(log_dir) / "losses.csv"
        self.entries = []
        self._header_written = False

    def log(self, epoch, step, **losses):
        self.entries.append({"epoch": epoch, "step": step, **losses})

    def flush(self):
        if not self.entries:
            return
        fieldnames = list(self.entries[0].keys())
        mode = "a" if self._header_written else "w"
        with open(self.log_path, mode, newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not self._header_written:
                writer.writeheader()
                self._header_written = True
            writer.writerows(self.entries)
        self.entries.clear()

    def plot_losses(self, output_path=None):
        if not self.log_path.exists():
            return
        epochs, d_losses, g_losses, l1_losses = [], [], [], []
        with open(self.log_path) as f:
            for row in csv.DictReader(f):
                epochs.append(int(row["epoch"]))
                d_losses.append(float(row.get("d_loss", 0)))
                g_losses.append(float(row.get("g_loss_adv", 0)))
                l1_losses.append(float(row.get("g_loss_l1", 0)))
        if not epochs:
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(epochs, d_losses, label="D Loss", alpha=0.7)
        ax1.plot(epochs, g_losses, label="G Adv Loss", alpha=0.7)
        ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
        ax1.set_title("Adversarial Losses"); ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, l1_losses, label="L1 Loss", color="green", alpha=0.7)
        ax2.set_xlabel("Step"); ax2.set_ylabel("Loss")
        ax2.set_title("Reconstruction Loss (L1)"); ax2.legend(); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        if output_path:
            plt.savefig(output_path, dpi=150, bbox_inches="tight")
        plt.show(); plt.close(fig)


print("Utilities defined ✓")

Utilities defined ✓


---
## 8. Download Dataset

Downloads the **CMP Facades** dataset (~29 MB) — a small paired dataset ideal for quick experiments. For larger/custom datasets, skip this cell and point `cfg.dataset_dir` to your data.

In [24]:
def download_facades(target_dir="./data/facades"):
    """Download and extract the Facades dataset (pix2pix format)."""
    target_dir = Path(target_dir)

    if (target_dir / "train").exists():
        n = len(list((target_dir / "train").glob("*")))
        print(f"Facades already exists at {target_dir} ({n} train images)")
        return

    target_dir.mkdir(parents=True, exist_ok=True)
    url = "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz"
    tar_path = target_dir / "facades.tar.gz"

    print(f"Downloading Facades dataset...")
    urllib.request.urlretrieve(url, tar_path)
    print("Extracting...")

    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(target_dir.parent)

    tar_path.unlink()
    n = len(list((target_dir / "train").glob("*")))
    print(f"Done! {n} training images at {target_dir}")


download_facades(cfg.dataset_dir)

Facades already exists at data/facades (400 train images)


---
## 9. Preview Dataset

Visualize a few training pairs to verify loading is correct.

In [25]:
# Load dataset and show samples
train_loader, val_loader = create_dataloaders(cfg)

label_batch, real_batch = next(iter(train_loader))
print(f"Label batch: {label_batch.shape}, range [{label_batch.min():.2f}, {label_batch.max():.2f}]")
print(f"Real batch:  {real_batch.shape}, range [{real_batch.min():.2f}, {real_batch.max():.2f}]")

# Display a few samples
n_show = min(4, label_batch.size(0))
fig, axes = plt.subplots(n_show, 2, figsize=(8, 4 * n_show))
for i in range(n_show):
    lbl_img = denormalize(label_batch[i]).permute(1, 2, 0).numpy().clip(0, 1)
    real_img = denormalize(real_batch[i]).permute(1, 2, 0).numpy().clip(0, 1)

    axes[i][0].imshow(lbl_img); axes[i][0].set_title("Label Map"); axes[i][0].axis("off")
    axes[i][1].imshow(real_img); axes[i][1].set_title("Real Photo"); axes[i][1].axis("off")

plt.suptitle("Training Pairs Preview", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

/tmp/ipykernel_98408/3278540087.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 10. Training Loop

The full adversarial training procedure:

1. **Update Discriminator**: Classify real pairs as real, fake pairs as fake
2. **Update Generator**: Fool D + minimize L1 distance to ground truth + (optional) feature matching

Training logs, visual samples, and checkpoints are saved periodically.

In [26]:
def train(cfg):
    """Main training function."""
    device = cfg.device
    print(f"{'='*60}")
    print(f"  Image Synthesis from Semantic Labeling — Training")
    print(f"{'='*60}")
    print(f"  Device:     {device}")
    print(f"  Image size: {cfg.image_size}×{cfg.image_size}")
    print(f"  Batch size: {cfg.batch_size}")
    print(f"  Epochs:     {cfg.epochs}")
    print(f"  Lambda L1:  {cfg.lambda_l1}")
    print(f"{'='*60}\n")

    # ── Data ───────────────────────────────────────────────────────
    train_loader, val_loader = create_dataloaders(cfg)

    # ── Models ─────────────────────────────────────────────────────
    generator = UNetGenerator(
        cfg.input_nc, cfg.output_nc, cfg.ngf, cfg.unet_depth, cfg.gen_dropout
    ).to(device)

    discriminator = PatchGANDiscriminator(
        cfg.input_nc, cfg.output_nc, cfg.ndf, cfg.n_layers_d
    ).to(device)

    g_params = sum(p.numel() for p in generator.parameters())
    d_params = sum(p.numel() for p in discriminator.parameters())
    print(f"Generator params:     {g_params:,}")
    print(f"Discriminator params: {d_params:,}\n")

    # ── Losses ─────────────────────────────────────────────────────
    criterion_gan = GANLoss(mode="vanilla").to(device)
    criterion_l1  = L1ReconstructionLoss(weight=cfg.lambda_l1)
    criterion_fm  = FeatureMatchingLoss(weight=cfg.lambda_fm) if cfg.use_feature_matching else None

    # ── Optimizers ─────────────────────────────────────────────────
    opt_g = torch.optim.Adam(generator.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))

    # ── Logger ─────────────────────────────────────────────────────
    logger = LossLogger(cfg.log_dir)

    # ── Fixed batch for consistent visual tracking ─────────────────
    fixed_labels, fixed_reals = next(iter(train_loader))
    fixed_labels = fixed_labels.to(device)
    fixed_reals  = fixed_reals.to(device)

    # ── Training Loop ──────────────────────────────────────────────
    history = {"epoch": [], "d_loss": [], "g_loss": []}
    global_step = 0

    for epoch in range(cfg.epochs):
        generator.train()
        discriminator.train()
        epoch_d, epoch_g = 0.0, 0.0
        t0 = time.time()

        for label_maps, real_images in train_loader:
            label_maps  = label_maps.to(device)
            real_images = real_images.to(device)

            # ── (1) Update Discriminator ───────────────────────────
            for _ in range(cfg.n_critic):
                opt_d.zero_grad()
                with torch.no_grad():
                    fake_images = generator(label_maps)

                _, pred_real = discriminator(label_maps, real_images)
                _, pred_fake = discriminator(label_maps, fake_images)

                loss_d = 0.5 * (criterion_gan(pred_real, True) + criterion_gan(pred_fake, False))
                loss_d.backward()
                opt_d.step()

            # ── (2) Update Generator ───────────────────────────────
            opt_g.zero_grad()
            fake_images = generator(label_maps)
            feats_fake, pred_fake = discriminator(label_maps, fake_images)

            loss_g_adv = criterion_gan(pred_fake, True)
            loss_g_l1  = criterion_l1(fake_images, real_images)

            loss_g_fm = torch.tensor(0.0, device=device)
            if criterion_fm is not None:
                feats_real, _ = discriminator(label_maps, real_images)
                loss_g_fm = criterion_fm(feats_real, feats_fake)

            loss_g = loss_g_adv + loss_g_l1 + loss_g_fm
            loss_g.backward()
            opt_g.step()

            epoch_d += loss_d.item()
            epoch_g += loss_g.item()
            global_step += 1

            logger.log(epoch=epoch, step=global_step,
                       d_loss=loss_d.item(), g_loss_adv=loss_g_adv.item(),
                       g_loss_l1=loss_g_l1.item(), g_loss_fm=loss_g_fm.item(),
                       g_loss_total=loss_g.item())

        # ── Epoch summary ──────────────────────────────────────────
        n = len(train_loader)
        elapsed = time.time() - t0
        avg_d, avg_g = epoch_d / n, epoch_g / n
        history["epoch"].append(epoch + 1)
        history["d_loss"].append(avg_d)
        history["g_loss"].append(avg_g)

        print(f"Epoch [{epoch+1:3d}/{cfg.epochs}]  D: {avg_d:.4f}  G: {avg_g:.4f}  ({elapsed:.1f}s)")
        logger.flush()

        # ── Save visual samples ────────────────────────────────────
        if (epoch + 1) % cfg.sample_interval == 0:
            generator.eval()
            with torch.no_grad():
                fixed_fakes = generator(fixed_labels)
            show_sample_grid(
                fixed_labels, fixed_fakes, fixed_reals,
                save_path=cfg.sample_dir / f"epoch_{epoch+1:04d}.png"
            )

        # ── Save checkpoint ────────────────────────────────────────
        if (epoch + 1) % cfg.save_interval == 0:
            save_checkpoint(
                generator, discriminator, opt_g, opt_d, epoch + 1,
                path=cfg.checkpoint_dir / f"checkpoint_{epoch+1:04d}.pth"
            )

    # ── Final save ─────────────────────────────────────────────────
    save_checkpoint(generator, discriminator, opt_g, opt_d, cfg.epochs,
                    path=cfg.checkpoint_dir / "latest.pth")

    print(f"\nTraining complete! Outputs at {cfg.output_dir}")
    return generator, discriminator, history


print("Training function defined ✓")
print("Run the next cell to start training.")

Training function defined ✓
Run the next cell to start training.


---
## 11. Run Training

Execute the training loop. Adjust `cfg.epochs` above if you want a shorter run.

In [27]:
generator, discriminator, history = train(cfg)


Training complete! Outputs at ./outputs


---
## 12. Loss Curves

In [28]:
# Plot training loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["epoch"], history["d_loss"], label="Discriminator", alpha=0.8)
ax1.plot(history["epoch"], history["g_loss"], label="Generator (total)", alpha=0.8)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training Loss Curves"); ax1.legend(); ax1.grid(True, alpha=0.3)

# Also plot from CSV for per-step detail
logger = LossLogger(cfg.log_dir)
logger.plot_losses(output_path=cfg.log_dir / "loss_curves.png")

/tmp/ipykernel_98408/1923582472.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close(fig)


---
## 13. Inference

Generate images from semantic label maps using the trained generator.

In [29]:
def run_inference(generator, data_loader, device, n_samples=8):
    """Generate and display results from the data loader."""
    generator.eval()
    all_labels, all_fakes, all_reals = [], [], []

    with torch.no_grad():
        for labels, reals in data_loader:
            labels = labels.to(device)
            fakes = generator(labels)
            all_labels.append(labels.cpu())
            all_fakes.append(fakes.cpu())
            all_reals.append(reals)
            if sum(l.size(0) for l in all_labels) >= n_samples:
                break

    all_labels = torch.cat(all_labels)[:n_samples]
    all_fakes  = torch.cat(all_fakes)[:n_samples]
    all_reals  = torch.cat(all_reals)[:n_samples]

    show_sample_grid(all_labels, all_fakes, all_reals, max_samples=n_samples, save_path=cfg.inference_dir / "final_results.png")
    return all_labels, all_fakes, all_reals


# ── Run inference on validation set (or train if no val) ───────────
eval_loader = val_loader if val_loader is not None else train_loader
print("Generating results...")
labels, fakes, reals = run_inference(generator, eval_loader, cfg.device, n_samples=8)

/tmp/ipykernel_98408/1923582472.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 14. Single Image Inference

Load and process a single semantic label map.

In [30]:
def infer_single(generator, image_path, image_size=256, device="cpu"):
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    if w > h * 1.5:
        img = img.crop((0, 0, w // 2, h))

    transform = transforms.Compose([
        transforms.Resize((image_size, image_size), Image.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    label = transform(img).unsqueeze(0).to(device)

    generator.eval()
    with torch.no_grad():
        fake = generator(label)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(label[0]).cpu().permute(1,2,0).numpy().clip(0,1))
    axes[0].set_title("Input Label Map"); axes[0].axis("off")
    axes[1].imshow(denormalize(fake[0]).cpu().permute(1,2,0).numpy().clip(0,1))
    axes[1].set_title("Generated Image"); axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(cfg.inference_dir / "single_test.png", dpi=150, bbox_inches="tight")  # ← save
    plt.close(fig)
    print(f"Saved to {cfg.inference_dir / 'single_test.png'}")

---
## Summary

This notebook implements a complete **Image Synthesis from Semantic Labeling** pipeline:

| Component | Implementation |
|-----------|---------------|
| **Generator** | U-Net with 8-level encoder-decoder + skip connections |
| **Discriminator** | PatchGAN (70×70 receptive field) |
| **Adversarial Loss** | Binary Cross-Entropy (vanilla GAN) |
| **Reconstruction Loss** | Weighted L1 pixel loss (λ=100) |
| **Feature Matching** | Optional L1 on discriminator features |
| **Dataset** | Paired image loader with augmentation |
| **Training** | Standard alternating G/D optimization |

All GAN components are **written entirely from scratch** using PyTorch primitives.

In [31]:
infer_single(generator, "data/facades/test/1.jpg", cfg.image_size, cfg.device)

Saved to outputs/inference/single_test.png


In [ ]:
with open("facade_painter.py", "w") as f:
    f.write(facade_code)
print("facade_painter.py written ✓")


import subprocess, sys

subprocess.Popen([
    sys.executable, "facade_painter.py",
    "--checkpoint", str(cfg.checkpoint_dir / "latest.pth")
])
print("Facade Painter launched! Check your taskbar.")